# 📝 LangGraph 과제 LV1: 서비스 의뢰 정리와 상품 설명 개선

**목표:** LLM과 Python 함수를 노드로 연결해 첫 상담 답변을 만들고, 대화가 이어질수록 상품 설명을 개선합니다. 교안 01의 State·Node·Edge·구조화 출력·Reducer를 두 업무에 적용합니다.

<img src="images/20_assignment_lv1_graph.png" width="1050" style="background:white" alt="실습 A는 의뢰에서 요구사항을 추출하고 미정 항목을 확인해 정리문을 만듭니다. 독립적인 실습 B는 수정 요청 입력과 상품 설명 수정을 반복합니다.">

| 실습 | State | 노드 | 엣지 | 실행 |
|---|---|---|---|---|
| A. 서비스 의뢰 정리 | 1번 | 2~4번 | 5번 | 6번 |
| B. 상품 설명 대화 수정 | 7번 | 8~9번 | 10번 | 10번 |

**작성 방법:** 1·7번은 State와 초기 입력 전체를 작성하고 나머지는 핵심 코드를 채웁니다. 긴 프롬프트·출력 스키마·입력 자료는 제공합니다. 두 실습은 서로 다른 가상 자료를 사용하며 A의 결과를 B에 전달하지 않습니다. 공통 import와 모델 객체는 이어 사용하므로 노트북은 위에서부터 순서대로 실행하세요.

**공통 규칙:** 노드는 변경할 필드만 반환합니다. 입력 State를 직접 바꾸지 말고 실행 결과를 새 변수로 받으세요. 확인 셀은 제공 코드입니다. 2·4번에서 정의한 LLM 노드는 6번에서, 8·9번 노드는 10번에서 실제로 실행합니다. 정의 직후의 입력 이름 출력만으로 구현이 맞다고 판단하지 마세요. 모델 답변은 문장이 똑같은지보다 원문 정보와 요청 조건을 지켰는지 확인합니다.

**모델 호출:** 6번 GPT 2회, 10번은 수정 요청마다 GPT 1회입니다. 지정한 두 요청이면 총 4회이며 `/종료`는 모델을 호출하지 않습니다. `OPENAI_CHAT_MODEL=gpt-5.6-luna`를 사용합니다.

## 준비: 서비스 의뢰와 모델

미정인 운영 조건을 임의로 확정하면 구현 범위가 달라집니다. 원문에서 확정 정보와 확인이 필요한 정보를 구분해 읽으세요.


In [ ]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, Markdown, display

material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)
print("사용 모델:", llm.model_name)

# 학습용 가상 서비스 의뢰: 정하지 않은 운영 조건도 포함
service_request = """
대학생을 위한 스터디룸 예약 서비스를 만들고 싶습니다.
학생들은 휴대폰 웹에서 이용하고, 운영자는 스터디룸 4개의 예약 현황을 확인합니다.
학생에게는 빈 시간 확인, 예약 신청, 내 예약 조회와 취소 기능이 필요합니다.
같은 방의 같은 시간에 중복 예약이 생기지 않아야 합니다.
한 번에 최대 몇 시간 예약할 수 있는지와 예약 시작 몇 시간 전까지 취소할 수 있는지는 아직 정하지 않았습니다.
출시일도 미정입니다.
""".strip()

# 키는 필수 확인 필드 이름, 값은 의뢰인에게 보여 줄 항목 이름
required_fields = {
    "max_hours_per_booking": "1회 최대 예약 시간",
    "cancellation_deadline_hours": "예약 시작 전 취소 가능 기한",
    "launch_date": "출시일",
}
print(service_request)


## 1. 서비스 의뢰를 공유할 State를 만듭니다

**배경**: 의뢰 원문, 추출한 요구사항, 미정 항목, 최종 정리문을 노드 사이에서 전달합니다.

**제공 코드**: service_request와 required_fields를 제공합니다. State와 초기 입력을 직접 작성합니다.

**작성 방식**: 전체 코드 작성

**요구사항**:

- `RequestState(TypedDict)`에 request_text: str, requirements: dict, missing_fields: list[str], brief: str를 선언하세요.
- `request_input`의 request_text에는 service_request를 넣습니다. requirements는 빈 딕셔너리, missing_fields는 빈 리스트, brief는 빈 문자열입니다.
- 지금 `request_input["requirements"]`는 빈 딕셔너리입니다. 6번에서 그래프를 실행하면 반환된 `request_result["requirements"]`에 `ServiceRequirements`의 필드 이름과 추출 정보 또는 `None`이 담깁니다. `request_input`은 그대로 유지됩니다.

**확인 기준**: 초기 입력에 원문이 있고 결과 세 필드는 비어 있습니다.

<details><summary>힌트</summary>

```text
TypedDict는 필드 이름과 타입을 선언하고, request_input은 실제 시작 데이터를 담습니다.
```

</details>


In [ ]:
from typing_extensions import TypedDict

# 여기에 코드를 작성하세요


필드 이름·타입과 초기값을 요구사항에 대조하세요. 원문 비교는 True여야 합니다.


In [ ]:
# [출력 확인]
print("State 필드:", list(RequestState.__annotations__))
print("원문이 같나요?", request_input["request_text"] == service_request)
print("초기 결과:", request_input["requirements"], request_input["missing_fields"], request_input["brief"])


## 2. LLM으로 요구사항을 추출합니다

**배경**: 의뢰의 자연어 표현을 코드가 읽을 수 있는 필드로 바꿉니다.

**제공 코드**: ServiceRequirements 스키마, 추출 프롬프트와 함수 골격을 제공합니다.

**작성 방식**: 코드 골격 완성

**요구사항**:

- requirements_chain은 requirements_prompt와 `llm.with_structured_output(ServiceRequirements, strict=True)`를 연결하세요.
- `extract_requirements`에서 `requirements_chain.invoke`를 호출하고 결과를 `result`에 담으세요. 입력은 `request_text` 키 하나를 가진 딕셔너리이며, 그 값은 `state["request_text"]`입니다.
- `result.model_dump()`를 requirements 값으로 반환하세요. 다른 State 필드는 반환하지 않습니다.
- 미정인 최대 예약 시간·취소 기한·출시일은 None입니다. 프롬프트와 스키마는 그대로 사용합니다.

**확인 기준**: 함수를 정의하는 단계입니다. 6번 실행 뒤 방 수 4와 요청 기능이 보존됐는지 확인합니다.

<details><summary>힌트</summary>

```text
체인은 Pydantic 객체를 반환합니다. model_dump()로 일반 딕셔너리로 바꿉니다.
```

</details>


In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# 제공: LLM이 추출할 요구사항의 형식
class ServiceRequirements(BaseModel):
    service_name: str = Field(description="의뢰에서 확인되는 서비스 종류")
    target_users: list[str] = Field(description="서비스를 이용할 대상")
    platform: str = Field(description="의뢰인이 말한 이용 환경")
    room_count: int = Field(description="운영할 스터디룸 수")
    features: list[str] = Field(description="의뢰에서 요청한 기능과 제약")
    max_hours_per_booking: int | None = Field(description="1회 최대 예약 시간. 미정이면 null")
    cancellation_deadline_hours: int | None = Field(description="예약 시작 몇 시간 전까지 취소 가능한지. 미정이면 null")
    launch_date: str | None = Field(description="의뢰인이 정한 출시일. 미정이면 null")

# 제공: 원문에 없는 조건을 확정하지 않는 추출 기준
requirements_prompt = ChatPromptTemplate.from_messages([
    ("system", "서비스 제작 의뢰에서 명시된 요구사항만 추출하세요. "
     "학생 기능, 운영자의 예약 현황 확인, 같은 방과 시간의 중복 예약 방지 조건을 보존하세요. "
     "미정인 예약 시간, 취소 기한, 출시일은 null로 반환하세요. "
     "통상적인 서비스 관행을 근거로 원문에 없는 결제, 로그인, 알림, 일정, 금액을 추가하지 마세요."),
    ("human", "{request_text}"),
])
requirements_chain = ...

def extract_requirements(state: RequestState):
    """의뢰를 구조화하고 딕셔너리로 변환한 요구사항만 반환합니다."""
    result = ...
    return ...


아래는 제공된 스키마와 프롬프트의 입력 이름 확인입니다. 작성한 추출 함수의 실제 반환값은 6번에서 확인합니다.


In [ ]:
# [출력 확인]
print("추출할 필드:", list(ServiceRequirements.model_fields))
print("입력 변수:", requirements_prompt.input_variables)


## 3. Python 노드로 미정 항목을 찾습니다

**배경**: None인지 확인하는 작업은 추출 결과에서 바로 판단할 수 있습니다.

**제공 코드**: required_fields의 키 순서는 최대 예약 시간, 취소 기한, 출시일입니다. 함수 골격을 제공합니다.

**작성 방식**: 코드 골격 완성

**요구사항**:

- missing_fields를 빈 리스트로 만들고 required_fields의 키를 차례로 순회하세요.
- `state["requirements"][field] is None`인 경우 그 field 이름을 추가하세요. 0이나 빈 문자열을 None으로 취급하지 않습니다.
- 반환값은 `missing_fields` 키 하나를 가진 딕셔너리입니다. 값에는 모은 필드 이름 리스트를 넣으세요. 입력 `state`는 바꾸지 않으며 LLM을 호출하지 않습니다.

**확인 기준**: 미정 필드는 required_fields에 선언한 순서로 담깁니다. 이 의뢰에서는 세 필드 모두 해당합니다.

<details><summary>힌트</summary>

```text
필드의 표시 이름이 아니라 키를 모읍니다. 안내용 이름 변환은 다음 노드가 담당합니다.
```

</details>


In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
def find_missing(state: RequestState):
    """필수 확인 필드 중 값이 None인 필드 이름을 반환합니다."""
    missing_fields = ...
    for field in required_fields:
        if ...:
            ...
    return ...


아래는 find_missing을 실제 호출하는 확인 셀입니다. 0과 빈 문자열은 남기고 None인 취소 기한만 선택해야 하며, 두 비교 출력은 True여야 합니다.


In [ ]:
# [출력 확인]
# 실제 함수를 호출하되 GPT는 사용하지 않습니다.
from copy import deepcopy

missing_test: RequestState = {
    "request_text": "확인용 입력", "requirements": {
        "max_hours_per_booking": 0, "cancellation_deadline_hours": None, "launch_date": "",
    }, "missing_fields": [], "brief": "기존 정리문",
}
missing_before = deepcopy(missing_test)
missing_update = find_missing(missing_test)
print("반환값:", missing_update)
print("None인 취소 기한만 선택했나요?", missing_update == {"missing_fields": ["cancellation_deadline_hours"]})
print("입력 전체가 유지됐나요?", missing_test == missing_before)


## 4. 첫 상담 답변을 작성하는 노드를 만듭니다

**배경**: 확정된 제작 범위를 정리하고, 정하지 않은 조건을 의뢰인에게 질문합니다.

**제공 코드**: 긴 프롬프트, 필드 이름 변환 코드, 함수 골격을 제공합니다.

**작성 방식**: 코드 골격 완성

**요구사항**:

- brief_chain은 brief_prompt와 llm을 연결하세요.
- `response`에는 `brief_chain.invoke`에 아래 입력 딕셔너리를 전달한 결과를 담으세요.
  - `request_text`: `state["request_text"]`의 원문 문자열
  - `requirements`: `state["requirements"]`를 `json.dumps`로 변환한 JSON 문자열. 한글을 읽기 쉽게 `ensure_ascii=False`로 설정합니다.
  - `missing_topics`: 제공 코드가 만든 `missing_topics` 리스트를 줄바꿈 문자로 연결한 문자열. `str.join`을 사용합니다.
- response.text를 brief 필드 하나로 반환하세요.
- 출력은 서비스 목표·확정 요구사항·확인 질문으로 구성합니다. 방 4개, 모바일 웹, 학생과 운영자 기능, 중복 예약 방지를 보존하고 미정 세 항목은 질문으로 남깁니다.

**확인 기준**: 임의의 운영 시간·취소 기한·출시일·개발 비용을 약속하지 않는 첫 상담 답변입니다.

<details><summary>힌트</summary>

```text
Node는 공유 State를 입력받지만 프롬프트에는 체인이 요구하는 입력 이름으로 전달합니다.
```

</details>


In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
import json

# 제공: 확인한 조건과 미정 조건을 구분하는 첫 상담 답변
brief_prompt = ChatPromptTemplate.from_messages([
    ("system", "서비스 제작 의뢰를 받은 담당자로서 첫 상담 답변을 한국어로 짧게 작성하세요. "
     "'서비스 목표', '확정 요구사항', '확인 질문'으로 나누세요. "
     "확정 요구사항에는 사용자, 모바일 웹, 방 수, 요청 기능, 중복 예약 방지를 빠뜨리지 마세요. "
     "확인 질문은 전달된 확인 항목에 대해 한 항목당 하나씩 작성하세요. "
     "null인 운영 규칙이나 출시일을 임의로 정하지 말고, 없는 기능·일정·비용을 약속하지 마세요."),
    ("human", "의뢰 원문:\n{request_text}\n추출한 요구사항:\n{requirements}\n확인할 항목:\n{missing_topics}"),
])
brief_chain = ...

def draft_brief(state: RequestState):
    """확정된 요구사항과 확인할 질문을 의뢰 정리문으로 작성합니다."""
    # 내부 필드 이름을 의뢰인이 읽을 항목 이름으로 변환
    missing_topics = [required_fields[field] for field in state["missing_fields"]]
    response = ...
    return ...


아래는 프롬프트 입력 이름 확인입니다. 작성한 함수의 실제 첫 상담 답변은 6번에서 원문과 대조합니다.


In [ ]:
# [출력 확인]
print("정리문 입력:", brief_prompt.input_variables)
print("확인 항목 이름:", list(required_fields.values()))


## 5. 세 노드를 고정 엣지로 연결합니다

**배경**: 정보 추출이 끝난 뒤 미정 항목을 확인하고, 두 결과를 읽어 정리문을 작성합니다.

**제공 코드**: 노드 등록 코드를 제공합니다.

**작성 방식**: 코드 골격 완성

**요구사항**:

- request_builder는 RequestState를 사용하는 StateGraph입니다.
- START -> extract_requirements -> find_missing -> draft_brief -> END로 연결하세요.
- request_graph에 request_builder.compile() 결과를 담으세요.

**확인 기준**: 세 작업이 고정 순서로 연결되며 compile 시점에는 LLM이 호출되지 않습니다.

<details><summary>힌트</summary>

```text
연결을 변경할 때는 새 builder 생성과 노드 등록부터 다시 실행합니다.
```

</details>


In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
from langgraph.graph import END, START, StateGraph

request_builder = ...
request_builder.add_node("extract_requirements", extract_requirements)
request_builder.add_node("find_missing", find_missing)
request_builder.add_node("draft_brief", draft_brief)

# 노드 간 실행 흐름 연결
...
request_graph = ...


그래프 그림의 노드와 화살표가 요구한 순서인지 확인하세요. 아직 모델을 호출하지 않습니다.


In [ ]:
# [출력 확인]
display(Image(request_graph.get_graph().draw_mermaid_png()))


## 6. 의뢰 정리 결과와 원문을 대조합니다

**배경**: 한 번의 invoke로 추출, 미정 확인, 첫 상담 답변을 이어 실행합니다.

**제공 코드**: 출력 코드를 제공합니다. 실제 GPT 호출은 이 문항에서 2회 발생합니다.

**작성 방식**: 코드 골격 완성

**요구사항**:

- request_result는 request_graph에 request_input을 invoke한 결과입니다.
- requirements의 room_count는 4, 세 미정 필드는 None이며 missing_fields는 required_fields의 키 순서와 같아야 합니다.
- brief가 요청한 기능과 제약을 보존하고, 미정 항목을 하나씩 묻는지 확인하세요. 표현은 달라도 됩니다.
- request_input의 원문과 빈 결과는 유지되어야 합니다.

**확인 기준**: 원문에 없는 확정 조건을 만들지 않고 세 확인 질문을 제시합니다.

<details><summary>힌트</summary>

```text
입력 State와 invoke의 반환 State는 다른 값입니다. 두 변수를 각각 출력해 비교합니다.
```

</details>


In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
request_result = ...
print("추출한 요구사항:")
print(json.dumps(request_result["requirements"], ensure_ascii=False, indent=2))
print("미정 필드:", request_result["missing_fields"])
display(Markdown(request_result["brief"]))


방 수 4, 미정 값 세 개 None, 두 비교 출력 True를 확인하세요. 기능·제약 보존과 세 확인 질문은 정리문을 직접 읽어 판단합니다.


In [ ]:
# [출력 확인]
print("방 수:", request_result["requirements"]["room_count"])
print("미정 값:", {key: request_result["requirements"][key] for key in required_fields})
print("미정 필드 순서가 맞나요?", request_result["missing_fields"] == list(required_fields))
print("처음 입력이 유지됐나요?", request_input == {
    "request_text": service_request, "requirements": {}, "missing_fields": [], "brief": "",
})


## 실습 B. 대화로 상품 설명을 개선합니다

새로운 상품 자료를 사용합니다. 첫 수정 요청의 말투와 제한을 두 번째 요청에서도 유지하며 상품 설명을 다듬습니다.

<img src="images/22_assignment_lv1_state_updates.png" width="1050" style="background:white" alt="current_description은 최신 설명으로 교체하고 messages는 add_messages로 요청과 답변을 누적합니다. product_text는 원본 정보를 유지합니다.">

**교체할 값과 쌓을 값:** current_description에는 최신 설명 하나만 보관하고, messages에는 요청과 답변을 계속 쌓습니다. 두 필드는 같은 노드에서 서로 다르게 갱신됩니다.


In [ ]:
# 학습용 가상 상품: 확인된 정보와 기존 설명을 제공
product_text = """
상품명: 포켓폴드 우산
방식: 수동 개폐식 3단 우산
무게: 230g / 접었을 때 길이: 24cm
색상: 네이비 / 기본 구성: 손목 스트랩
사용 후 펼쳐서 말린 뒤 보관하고, 강풍에는 사용을 피하세요.
방풍 등급, 자외선 차단 성능, 인증, 보증 기간에 관한 자료는 없습니다.
""".strip()
initial_description = (
    "포켓폴드 우산은 직접 여닫는 수동 개폐식 3단 우산입니다. "
    "무게는 230g이고 접으면 24cm이며, 네이비 색상에 손목 스트랩이 기본으로 포함됩니다. "
    "사용 후 펼쳐서 말린 뒤 보관하고 강풍에는 사용을 피하세요."
)
print(product_text)
print("수정 전 설명:", initial_description)


## 7. 상품 원문·최신 설명·대화를 담을 State를 만듭니다

**배경**: 변하지 않는 상품 정보, 계속 고치는 설명, 누적할 대화를 구분합니다.

**제공 코드**: product_text와 initial_description, 관련 import를 제공합니다. State와 초기 입력을 작성합니다.

**작성 방식**: 전체 코드 작성

**요구사항**:

- CopyState(TypedDict)에 product_text: str, current_description: str, messages: Annotated[list[AnyMessage], add_messages]를 선언하세요.
- `copy_input`은 다음 세 키를 가진 초기 입력 딕셔너리입니다.
  - `product_text`: 제공된 `product_text` 원문
  - `current_description`: 제공된 `initial_description`
  - `messages`: 빈 리스트
- current_description에는 Reducer를 지정하지 않습니다. 새 문자열로 교체하는 기본 규칙을 사용합니다.

**확인 기준**: 새 설명은 교체되고 요청·답변은 누적되도록 필드가 선언됩니다.

<details><summary>힌트</summary>

```text
Annotated는 messages 필드에 add_messages 갱신 규칙을 연결합니다.
```

</details>


In [ ]:
from typing import Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

# 여기에 코드를 작성하세요


messages의 add_messages 선언, 빈 초기 대화, 시작 설명을 요구사항과 대조하세요. 실제 누적·교체는 10번에서 확인합니다.


In [ ]:
# [출력 확인]
print("messages 선언:", CopyState.__annotations__["messages"])
print("초기 대화:", copy_input["messages"])
print("시작 설명:", copy_input["current_description"])


## 8. 수정 요청을 입력받는 노드를 만듭니다

**배경**: 사용자의 요청을 HumanMessage로 바꿔 대화에 전달합니다.

**제공 코드**: 함수 골격, 출력문, 종료 경로를 선택하는 route_copy는 제공합니다.

**작성 방식**: 코드 골격 완성

**요구사항**:

- request는 `input("수정 요청 (/종료로 끝내기): ").strip()`의 결과입니다.
- read_request는 `{"messages": [HumanMessage(content=request)]}`를 반환하세요.
- `/종료`도 대화에는 남지만 route_copy가 종료 경로로 보내므로 LLM을 호출하지 않습니다.

**확인 기준**: 입력 하나를 메시지 하나로 반환합니다. 기존 메시지 목록을 직접 수정하지 않습니다.

<details><summary>힌트</summary>

```text
노드는 messages 갱신을 반환하고 라우터는 계속 또는 종료라는 경로 이름을 반환합니다.
```

</details>


In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
from langchain_core.messages import HumanMessage

def read_request(state: CopyState):
    """수정 요청을 입력받고 사용자 메시지 한 개를 반환합니다."""
    request = ...
    print("사용자 입력:", request)
    return ...

# 제공: 마지막 입력에 따라 계속 수정하거나 종료
# input()으로 한 번의 invoke 안에서 대화하며 별도 체크포인터는 사용하지 않음
def route_copy(state: CopyState):
    if state["messages"][-1].text == "/종료":
        return "종료"
    return "계속"


아래는 제공된 라우터 확인입니다. 일반 요청은 계속, 종료 요청은 종료여야 합니다. 작성한 read_request의 입력 처리는 10번에서 확인합니다.


In [ ]:
# [출력 확인]
print("일반 요청:", route_copy({**copy_input, "messages": [HumanMessage(content="짧게 써줘")]}))
print("종료 요청:", route_copy({**copy_input, "messages": [HumanMessage(content="/종료")]}))


## 9. 최신 설명을 교체하고 답변을 누적합니다

**배경**: 원문을 지키면서 현재 설명과 앞선 수정 요청을 함께 반영합니다.

**제공 코드**: 프롬프트, 출력문, 함수 골격을 제공합니다.

**작성 방식**: 코드 골격 완성

**요구사항**:

- copy_chain은 copy_prompt와 llm을 연결하세요.
- `copy_chain.invoke`에 `product_text`, `current_description`, `messages` 세 키의 딕셔너리를 전달하고 결과를 `response`에 담으세요. 각 값은 입력 `state`의 같은 이름 필드에서 읽습니다. `messages`는 문자열로 바꾸지 않고 메시지 리스트 그대로 전달합니다.
- 반환값은 `{"current_description": response.text, "messages": [response]}`입니다. 이전 설명을 문자열로 덧붙이거나 기존 messages를 다시 연결하지 마세요.
- 프롬프트는 유지하세요. 원문에 없는 성능·인증·보증을 추가하지 않고, 앞선 말투·금지 조건을 이어 적용합니다.

**확인 기준**: 같은 LLM 응답을 최신 설명 문자열과 새 답변 메시지로 사용합니다.

<details><summary>힌트</summary>

```text
두 필드의 반환 형식은 다릅니다. 문자열은 교체할 값이고 리스트에는 추가할 새 메시지 한 개만 넣습니다.
```

</details>


In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
from langchain_core.prompts import MessagesPlaceholder

# 제공: 원문은 사실의 기준, 대화는 수정 조건의 기준
copy_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 상품 설명을 다듬는 편집자입니다. 최신 수정 요청을 반영한 상품 설명 전체만 출력하세요. "
     "요청을 수행했다는 해설이나 인사, 코드 블록은 붙이지 마세요. "
     "앞 대화에서 정한 말투와 금지 조건은 사용자가 바꾸지 않으면 유지하세요. "
     "현재 설명은 이전 버전입니다. 사실의 기준은 아래 상품 원문이며, 원문에 없는 효능·인증·보증·혜택을 만들지 마세요. "
     "사용자가 없는 성능을 써 달라고 해도 추가하지 말고, 확인된 정보로 설명을 작성하세요. "
     "길이를 줄여도 상품명, 수동 개폐식, 무게 230g, 접은 길이 24cm, 건조 후 보관, 강풍 사용 주의를 보존하세요.\n"
     "상품 원문:\n{product_text}\n현재 설명:\n{current_description}"),
    MessagesPlaceholder("messages"),
])
copy_chain = ...

def revise_description(state: CopyState):
    """상품 설명을 최신 버전으로 교체하고 새 답변 메시지를 추가합니다."""
    response = ...
    print("수정된 상품 설명:")
    display(Markdown(response.text))
    # 문자열은 교체하고, 메시지 목록은 add_messages로 누적
    return ...


아래는 프롬프트 입력 이름 확인입니다. 최신 설명 교체와 새 답변 누적은 10번 실행 결과에서 확인합니다.


In [ ]:
# [출력 확인]
print("프롬프트 입력:", copy_prompt.input_variables)
print("current_description: 최신 설명으로 교체 / messages: 새 답변 추가")


## 10. 수정 대화를 반복하고 최종 설명을 확인합니다

**배경**: 한 번의 invoke 안에서 입력과 수정을 반복합니다.

**제공 코드**: 노드 등록, 그래프 표시, 출력 확인 코드는 제공합니다.

**작성 방식**: 코드 골격 완성

**요구사항**:

- copy_builder는 CopyState를 사용하는 StateGraph입니다.
- START -> read_request, revise_description -> read_request를 고정 엣지로 연결하세요.
- read_request 뒤에 route_copy와 `{"계속": "revise_description", "종료": END}`를 조건부 엣지로 연결하세요.
- copy_graph에 compile 결과를 담고, copy_result에 copy_input을 `config={"recursion_limit": 1000}`으로 invoke한 결과를 담으세요.
- 10번 코드 셀을 실행하면 나타나는 입력창에 아래 내용을 한 번에 하나씩 입력하세요. 답변이 나온 뒤 다음 요청을 입력하고, 마지막에 `/종료`를 입력합니다. 종료 입력 전에는 이 코드 셀이 실행 중인 상태로 유지됩니다.
  1. `선물용으로 차분하게 소개해줘. 앞으로 최고, 완벽 같은 최상급 표현과 이모지는 쓰지 마.`
  2. `그 조건을 유지하면서 정확히 세 문장으로 줄여줘. 제목과 번호 없이 문장마다 줄을 바꾸고, 각 문장은 마침표로 끝내줘.`
  3. `/종료`
- 최종 설명은 상품명·수동 개폐·230g·24cm·건조 후 보관·강풍 주의를 유지해야 합니다. 원문에 없는 인증이나 성능을 추가하면 안 됩니다.
- messages는 human, ai, human, ai, human 순서의 5개입니다. 최신 설명은 두 번째 AI 답변과 같아야 하며, 첫 번째와 두 번째 AI 답변이 모두 대화에 남아야 합니다. 두 답변의 문장이 서로 달라야 하는 것은 아닙니다.
- copy_input의 messages는 빈 리스트, current_description은 initial_description으로 유지되어야 합니다.

**확인 기준**: 최신 설명은 세 문장이 되고, 이전 말투·금지 조건과 대화 기록은 유지됩니다. /종료에는 답변하지 않습니다. 그래프 구조·메시지 수가 맞더라도 생성 문장이 조건을 놓치면 어느 조건이 빠졌는지 별도로 적으세요.

<details><summary>힌트</summary>

```text
누적된 대화에서 AI 답변 두 개를 찾아 최신 문자열과 비교합니다. 처음 입력과 최종 결과를 혼동하지 않습니다.
```

</details>


In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
copy_builder = ...
copy_builder.add_node("read_request", read_request)
copy_builder.add_node("revise_description", revise_description)

# 입력 후 수정하거나 종료하고, 수정 뒤에는 다음 입력으로 이동
...
copy_graph = ...
display(Image(copy_graph.get_graph().draw_mermaid_png()))

copy_result = ...


메시지 5개와 human·ai·human·ai·human 순서, 두 비교 출력 True를 확인하세요. 두 AI 답변의 상품 사실·말투·금지 표현을 확인하고, 최종 답변은 세 문장인지 확인하세요.


In [ ]:
# [출력 확인]
print("메시지 수:", len(copy_result["messages"]))
print("메시지 역할:", [message.type for message in copy_result["messages"]])
for index, message in enumerate(copy_result["messages"], start=1):
    print(f"[{index}] {message.type}")
    for line in message.text.splitlines():
        print("  ", line)
    print()
print("최종 설명:")
display(Markdown(copy_result["current_description"]))
print("최신 답변과 같나요?", copy_result["current_description"] == copy_result["messages"][-2].text)
print("처음 입력이 유지됐나요?", copy_input["messages"] == [] and copy_input["current_description"] == initial_description)
